# Pipeline de cartas Pokémon en Google Colab

Este cuaderno descarga el subconjunto público de 1.000 imágenes desde Kaggle, lo copia a `img/` y ejecuta el pipeline completo. Los scripts y pesos finales proceden del repositorio de GitHub; Google Drive puede usarse opcionalmente para reemplazar los pesos.

## Antes de ejecutar

1. Publica este proyecto en GitHub y cambia `GITHUB_REPOSITORY` en la celda siguiente.
2. Para usar pesos alternativos, súbelos a Google Drive y activa `USE_DRIVE_MODELS`.
3. En Colab, selecciona **Entorno de ejecución > Cambiar tipo de entorno de ejecución > T4 GPU** para usar `DEVICE = '0'`.

In [ ]:
# --- Configuración modificable ---
GITHUB_REPOSITORY = 'https://github.com/r041292/Pk-TCG-illustration-segmentation.git'  # Cámbiala
GITHUB_BRANCH = 'main'
PROJECT_DIR = '/content/pokemon-card-pipeline'
USE_DRIVE_MODELS = False  # True para reemplazar los pesos versionados con pesos de Drive
DRIVE_MODELS_DIR = '/content/drive/MyDrive/pokemon-card-models'  # Solo si USE_DRIVE_MODELS = True

# Nombres de los pesos dentro de DRIVE_MODELS_DIR. Cámbialos si son distintos.
OBB_MODEL_NAME = 'roboflow_obb_20260825_134802_best.pt'
CLASSIFIER_MODEL_NAME = 'best_classifier.pt'
SEGMENTATION_MODEL_NAME = 'ilustracion_ventana_seg_20260825_191947_best.pt'

LIMIT = 0                 # 0 procesa todas las imágenes
DEVICE = '0'              # '0' para GPU de Colab; 'cpu' si no hay GPU


In [ ]:
# Monta Drive, descarga los scripts desde GitHub e instala dependencias.
from google.colab import drive
from pathlib import Path
import shutil
import subprocess
import sys

if USE_DRIVE_MODELS:
    drive.mount('/content/drive')
if 'USUARIO/REPOSITORIO' in GITHUB_REPOSITORY:
    raise ValueError('Cambia GITHUB_REPOSITORY por la URL real de tu repositorio.')

project = Path(PROJECT_DIR)
if project.exists():
    shutil.rmtree(project)
subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--depth', '1', GITHUB_REPOSITORY, str(project)], check=True)
# Do not install requirements.txt here: it also contains optional RealESRGAN/SwinIR packages.
# The core pipeline only needs the lightweight dependency set below.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project / 'requirements_pipeline.txt'), 'kagglehub'], check=True)


In [ ]:
# Usa los pesos incluidos en el repositorio o pesos alternativos de Drive.
if USE_DRIVE_MODELS:
    models_dir = Path(DRIVE_MODELS_DIR)
    obb_model = models_dir / OBB_MODEL_NAME
    classifier_model = models_dir / CLASSIFIER_MODEL_NAME
    segmentation_model = models_dir / SEGMENTATION_MODEL_NAME
else:
    obb_model = project / 'models/card_obb/roboflow_obb_20260825_134802_best.pt'
    classifier_model = project / 'models/artwork_classifier/tipo_ilustracion_v1_best.pt'
    segmentation_model = project / 'models/illustration_segmentation/ilustracion_ventana_seg_20260825_191947_best.pt'

for model in (obb_model, classifier_model, segmentation_model):
    if not model.is_file():
        raise FileNotFoundError(f'No se encontró el peso: {model}')
    print('OK:', model)


In [ ]:
# Descarga el subconjunto público de 1.000 imágenes de Kaggle y lo prepara en img/.
import kagglehub

dataset_root = Path(kagglehub.dataset_download('rubielvelasquez/pokemon-tcg-real-card-images-1k-subset'))
source_images = dataset_root
print('Dataset descargado en:', dataset_root)

input_dir = project / 'img'

# Limpiar la entrada y los resultados evita mezclar archivos de ejecuciones anteriores.
for folder_name in ('img', 'img_pre', 'img_segm_yolo', 'img_refined', 'img_clasif'):
    folder = project / folder_name
    if folder.exists():
        shutil.rmtree(folder)
input_dir.mkdir(parents=True)

extensions = {'.jpg', '.jpeg', '.png', '.webp'}
images = [path for path in source_images.rglob('*') if path.is_file() and path.suffix.lower() in extensions]
if not images:
    raise FileNotFoundError(f'La carpeta {source_images} no contiene imágenes compatibles.')
for image in images:
    shutil.copy2(image, input_dir / image.name)

print(f'{len(images)} imágenes copiadas de {source_images} a {input_dir}')


In [ ]:
# Ejecuta las cuatro etapas: img -> img_pre -> img_segm_yolo -> img_refined -> img_clasif
command = [
    sys.executable, str(project / 'run_pipeline.py'),
    '--limit', str(LIMIT),
    '--device', DEVICE,
    '--obb-model', str(obb_model),
    '--classifier-model', str(classifier_model),
    '--segmentation-model', str(segmentation_model),
]
subprocess.run(command, cwd=project, check=True)


In [ ]:
# Guardar un ZIP de los resultados; se usa Drive solo si está montado.
output_base = Path('/content/drive/MyDrive/pokemon-card-results/img_clasif') if USE_DRIVE_MODELS else Path('/content/img_clasif')
output_base.parent.mkdir(parents=True, exist_ok=True)
zip_path = shutil.make_archive(str(output_base), 'zip', project / 'img_clasif')
print('Resultados guardados en:', zip_path)
